# Measuring a Classifier Honestly: Precision, Recall, Thresholds and Calibration

The full code of [Measuring a Classifier Honestly: Precision, Recall, Thresholds and Calibration](https://residual-stream.lvtan71.workers.dev/posts/measuring-a-classifier/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/measuring-a-classifier) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/measuring-a-classifier.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "measuring-a-classifier"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `measure.py`

The confusion matrix of one split; accuracy, precision, recall, F1, ROC-AUC and average precision of the three routers over 800 pooled test predictions; the threshold sweep; three ways to pick a threshold when a miss costs twice a false alarm.

In [ ]:
!{sys.executable} measure.py

The post printed:

```text
split 0 test: TP 6  FP 2  FN 4  TN 28
no penalty     acc 0.833  precision 0.795  recall 0.445  F1 0.571  ROC-AUC 0.861  AP 0.714
L2             acc 0.809  precision 0.776  recall 0.330  F1 0.463  ROC-AUC 0.850  AP 0.688
L2 + keywords  acc 0.885  precision 0.891  recall 0.615  F1 0.728  ROC-AUC 0.888  AP 0.790
threshold 0.1: precision 0.563  recall 0.830  cost 197
threshold 0.2: precision 0.750  recall 0.750  cost 150
threshold 0.3: precision 0.789  recall 0.710  cost 154
threshold 0.4: precision 0.830  recall 0.660  cost 163
threshold 0.5: precision 0.891  recall 0.615  cost 169
threshold 0.6: precision 0.918  recall 0.560  cost 186
threshold 0.7: precision 0.934  recall 0.495  cost 209
threshold 0.8: precision 0.931  recall 0.405  cost 244
test cost at 0.5: 169;  theory's 1/3: 156;  tuned on validation: 167;  tuned on test itself: 149
```

## `calibration.py`

Reliability tables and ECE of two routers, the ECE floor of perfectly calibrated labels, and temperature scaling fit on each split's validation set.

In [ ]:
!{sys.executable} calibration.py

The post printed:

```text
no penalty: ECE 0.109, Brier 0.134
   p in [0.0, 0.2): 632 predictions, mean p 0.03, actually serving 0.12
   p in [0.2, 0.4):  40 predictions, mean p 0.28, actually serving 0.62
   p in [0.4, 0.6):  34 predictions, mean p 0.50, actually serving 0.79
   p in [0.6, 0.8):  33 predictions, mean p 0.72, actually serving 0.64
   p in [0.8, 1.0):  61 predictions, mean p 0.92, actually serving 0.84
L2 + keywords: ECE 0.037, Brier 0.094
   p in [0.0, 0.2): 600 predictions, mean p 0.06, actually serving 0.08
   p in [0.2, 0.4):  41 predictions, mean p 0.31, actually serving 0.44
   p in [0.4, 0.6):  37 predictions, mean p 0.49, actually serving 0.54
   p in [0.6, 0.8):  35 predictions, mean p 0.71, actually serving 0.89
   p in [0.8, 1.0):  87 predictions, mean p 0.94, actually serving 0.93
ECE of perfectly calibrated labels: mean 0.016, 95th percentile 0.027
no penalty     tau median 1.99 (range 1.03-3.08):  ECE 0.109 -> 0.031
L2             tau median 1.14 (range 0.73-1.67):  ECE 0.061 -> 0.043
L2 + keywords  tau median 1.24 (range 0.28-2.03):  ECE 0.037 -> 0.027
```

## `vietnamese.py`

The routers on 20 Vietnamese questions they never trained on: which words the English vocabulary sees, recall, false alarms and per-question catch rates over 20 splits.

In [ ]:
!{sys.executable} vietnamese.py

The post printed:

```text
p=0.94 caught  seen=['kv', 'cache', 'a', '70b']  KV cache của mô hình 70B lớn cỡ nào?
p=0.98 caught  seen=['i', 'decode', 'prefill', 'i', 'token']  Tại sao decode chậm hơn prefill trên mỗi token?
p=0.07 MISSED  seen=['i', 'd']  Làm sao phục vụ một mô hình cho nhiều người dùng cùng lúc?
p=0.17 MISSED  seen=['token']  Vì sao token đầu tiên ra chậm như vậy?
p=0.08 MISSED  seen=['a']  Lượng tử hóa ảnh hưởng thế nào đến độ trễ?
p=0.39 MISSED  seen=['gpu', 'i']  Một GPU H100 chạy được bao nhiêu yêu cầu đồng thời?
p=0.59 caught  seen=['continuous', 'batching']  Continuous batching tăng thông lượng ra sao?
p=0.89 caught  seen=['vllm', 'd', 'paged', 'attention']  vLLM có dùng paged attention không?
p=0.07 MISSED  seen=['i', 'i', 'i', 'd']  Tại sao máy chủ chậm lại khi có thêm người dùng?
p=0.31 MISSED  seen=['speculative', 'decoding', 'i']  Speculative decoding tiết kiệm được bao nhiêu thời gian?
no penalty     recall 0.735  false-alarm rate 0.065  per-question catch rate: 0.85 1.00 0.50 1.00 0.00 0.95 0.75 0.80 0.50 1.00
L2             recall 0.505  false-alarm rate 0.000  per-question catch rate: 0.65 1.00 0.05 0.65 0.00 0.80 0.55 0.55 0.05 0.75
L2 + keywords  recall 0.540  false-alarm rate 0.000  per-question catch rate: 1.00 1.00 0.00 0.00 0.00 0.95 1.00 1.00 0.00 0.45
```